# Multi-Stop Route Planner Agent — Punjab OpenStreetMap Demo
**CSE476 CA1 Project 1 — Topic T14 (Multi-Stop Route Planner)**

This notebook demonstrates a **real agentic plan-act loop** using real-world geographical data extracted from the **`punjab.pbf` OpenStreetMap dataset**.

### Agentic Criteria Met:
1. **Two Tools Called**: `get_distance(a, b)` (Haversine formula on OSM coordinates with distance caching) and `order_stops(start, stops)` (Greedy nearest-neighbor loop + 2-opt path optimization pass).
2. **Plan-Act Loop**: Multi-step decision trace where each step selects the optimal stop based on tool outputs.
3. **Session Memory**: Remembers visited locations across turns, continues from the previous trip's endpoint, and caches calculated distances.

In [1]:
from agent import RouteAgent, print_trace, LOCATIONS

print(f"Loaded {len(LOCATIONS)} real Punjab locations from punjab.pbf dataset.")
agent = RouteAgent()

Loaded 15670 real Punjab locations from punjab.pbf dataset.


## Example Goal 1 — "Plan an efficient trip to Jalandhar, Ludhiana, Amritsar, Mohali starting from Phagwara"

The agent evaluates candidates dynamically using `get_distance` and applies a 2-opt repair pass to optimize the overall Punjab travel itinerary.

In [2]:
goal_1 = ["Jalandhar", "Ludhiana", "Amritsar", "Mohali"]
result_1 = agent.plan_trip(goal_1, start="Phagwara")
print_trace(result_1)


📍 Start: Phagwara
------------------------------------------------------------
  Step 1: at Phagwara
          Evaluated distances: [Jalandhar=22.13km, Ludhiana=35.52km, Amritsar=96.30km, Mohali=107.41km]
          ➔ Decision: go to Jalandhar (nearest at 22.13 km)
  Step 2: at Jalandhar
          Evaluated distances: [Ludhiana=53.85km, Amritsar=74.30km, Mohali=129.54km]
          ➔ Decision: go to Ludhiana (nearest at 53.85 km)
  Step 3: at Ludhiana
          Evaluated distances: [Amritsar=122.79km, Mohali=85.64km]
          ➔ Decision: go to Mohali (nearest at 85.64 km)
  Step 4: at Mohali
          Evaluated distances: [Amritsar=203.58km]
          ➔ Decision: go to Amritsar (nearest at 203.58 km)
------------------------------------------------------------
🛣️  Final Route: Phagwara -> Jalandhar -> Amritsar -> Ludhiana -> Mohali
📏 Total Distance: 304.85 km



## Example Goal 2 — "Now visit Jalandhar, Patiala, Moga, Bathinda"

**Memory Demonstration**:
- **Memory in Action**: `Jalandhar` was visited in Goal 1, so the agent automatically skips it.
- **Location Continuity**: The trip starts automatically at `Mohali` (the endpoint of Goal 1).

In [3]:
goal_2 = ["Jalandhar", "Patiala", "Moga", "Bathinda"]
result_2 = agent.plan_trip(goal_2)
print_trace(result_2)

assert result_2["start"] == "Mohali", "Should continue from the last trip endpoint (Mohali)"
assert "Jalandhar" in result_2["skipped_already_visited"], "Jalandhar should be skipped as it was visited in Goal 1"
print("PASS: Memory check passed — Jalandhar was skipped and trip continued from Mohali.")


📍 Start: Mohali
⚠️  Skipped (Already Visited): ['Jalandhar']
------------------------------------------------------------
  Step 1: at Mohali
          Evaluated distances: [Patiala=53.05km, Moga=147.72km, Bathinda=177.56km]
          ➔ Decision: go to Patiala (nearest at 53.05 km)
  Step 2: at Patiala
          Evaluated distances: [Moga=142.13km, Bathinda=149.34km]
          ➔ Decision: go to Moga (nearest at 142.13 km)
  Step 3: at Moga
          Evaluated distances: [Bathinda=71.81km]
          ➔ Decision: go to Bathinda (nearest at 71.81 km)
------------------------------------------------------------
🛣️  Final Route: Mohali -> Patiala -> Moga -> Bathinda
📏 Total Distance: 266.98 km

PASS: Memory check passed — Jalandhar was skipped and trip continued from Mohali.


## Example Goal 3 — "Visit Rupnagar and Hoshiarpur, then return to start"

Demonstrates the **Group of 3 add-on: Return-to-Start option**.

In [4]:
goal_3 = ["Rupnagar", "Hoshiarpur"]
result_3 = agent.plan_trip(goal_3, return_to_start=True)
print_trace(result_3)

assert result_3["route"][0] == result_3["route"][-1], "Route should end at starting location"
print("PASS: Return-to-start check passed.")


📍 Start: Bathinda
------------------------------------------------------------
  Step 1: at Bathinda
          Evaluated distances: [Rupnagar=173.33km, Hoshiarpur=173.41km]
          ➔ Decision: go to Rupnagar (nearest at 173.33 km)
  Step 2: at Rupnagar
          Evaluated distances: [Hoshiarpur=86.29km]
          ➔ Decision: go to Hoshiarpur (nearest at 86.29 km)
------------------------------------------------------------
🛣️  Final Route: Bathinda -> Rupnagar -> Hoshiarpur -> Bathinda
📏 Total Distance: 433.03 km

PASS: Return-to-start check passed.


## Session Summary Report (Group-of-3 Add-on)

A cumulative total-distance report aggregated across all planned legs in the session.

In [5]:
summary = agent.summary_report()
summary